<a href="https://colab.research.google.com/github/nimishac28/AI-agent-Transaction/blob/main/AI_agent_gmail_tracker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-openai pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 18.3 MB/s eta 0:00:00


In [ ]:
!pip install -q google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
import re

SENSITIVE_PATTERNS = [
    r"\botp\b",
    r"\bone[-\s]?time\s+password\b",
    r"\bverification\s+code\b",
    r"\bsecurity\s+code\b",
    r"\bpassword\b",
    r"\bpasscode\b",
    r"\bauthentication\s+code\b",
    r"\blogin\s+code\b",
    r"\bprofile\s+summary\b",
    r"\bagreement\b",
    r"\bauth\w*\b",
    r"\bverification\w*\b",]

SUCCESSFUL_TRANSACTION_PATTERNS = [
    r"\btransaction\b",
    r"\btransaction\s+(?:was\s+)?successful\b",
    r"\bpayment\s+(?:was\s+)?successful\b",
    r"\bcard\s+(?:was\s+)?used\b",
    r"\bwas\s+used\b",
    r"\bhas\s+(?:just\s+)?been\s+swiped\b",
    r"\bswiped\b",
    r"\bspent\b",
    r"\bpurchase\b",
    r"\bcharged\b",
]

NON_TRANSACTION_PATTERNS = [
    # Loans and credit offers
    r"\bloan\b",
    r"\bloan\s+(?:amount|offer|update|available)\b",
    r"\bcredit\s+limit\b",

    # Statements and bills
    r"\b(?:credit\s+card\s+)?statement\b",
    r"\bbill\b",
    r"\bbill\s+payment\b",
    r"\bminimum\s+(?:amount\s+)?due\b",
    r"\bpayment\s+due\b",

    # Marketing / promotional emails
    r"\boffer(?:s)?\b",
    r"\bpromotion(?:al)?\b",
    r"\bpre[-\s]?approved\b",
]

def is_sensitive_email(subject: str, body: str) -> bool:
    """
    Returns True if the email contains authentication credentials
    or other sensitive security-related information.
    """

    text = f"{subject} {body}"

    return any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in SENSITIVE_PATTERNS
    )


user_cards = input(
    "Enter the banks/cards you want to track (comma-separated): "
)

TRACKED_CARDS = [
    card.strip().lower()
    for card in user_cards.split(",")
]

print("Tracking:", TRACKED_CARDS)

def build_card_keywords(TRACKED_CARDS: str):
    """
    Create common keyword variations for a tracked card.
    """
    card = TRACKED_CARDS.strip().lower()

    return [
        card,
        f"{card} credit card"
    ]


CARD_KEYWORDS = {
    card: build_card_keywords(card)
    for card in TRACKED_CARDS
}


def get_tracked_card(subject: str, body: str):
    text = f"{subject} {body}".lower()

    for card_name, config in TRACKED_CARDS.items():
        for keyword in config["keywords"]:
            if keyword.lower() in text:
                return card_name

    return None

def is_tracked_card_email(subject: str, body: str) -> bool:
    """
    Returns True only if the email belongs to one of the
    user's configured cards/banks.
    """

    text = f"{subject} {body}".lower()

      # 🚫 First reject known non-purchase emails
    if any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in NON_TRANSACTION_PATTERNS
    ):
        return False

    # ✅ Then look for actual spending activity
    return any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in SUCCESSFUL_TRANSACTION_PATTERNS
    )

def is_successful_transaction(subject: str, body: str) -> bool:
    text = f"{subject} {body}"

    return any(
        re.search(pattern, text, re.IGNORECASE)
        for pattern in SUCCESSFUL_TRANSACTION_PATTERNS
    )

def should_process_transaction_email(subject: str, body: str) -> bool:
    # Never process sensitive/security emails
    if is_sensitive_email(subject, body):
        return False

    # Only process emails related to cards selected by the user
    if not is_tracked_card_email(subject, body):
        return False

    # Only process successful transactions
    if not is_successful_transaction(subject, body):
        return False

    return True

from pydantic import BaseModel, Field
from typing import Optional


class CreditCardTransaction(BaseModel):
    bank_name: Optional[str] = Field(
        default=None,
        description="Bank or credit card issuer mentioned in the email"
    )

    card_last_four: Optional[str] = Field(
        default=None,
        description="Last four digits of the credit card, if explicitly mentioned"
    )

    amount: Optional[float] = Field(
        default=None,
        description="Transaction amount as a number without currency symbols"
    )

    merchant: Optional[str] = Field(
        default=None,
        description="Merchant where the card was used"
    )

    transaction_date: Optional[str] = Field(
        default=None,
        description="Transaction date if explicitly mentioned in the email"
    )




Enter the banks/cards you want to track (comma-separated): hdfc, rbl
Tracking: ['hdfc', 'rbl']


In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0
)

structured_llm = llm.with_structured_output(CreditCardTransaction)

In [ ]:
def extract_transaction(subject: str, body: str):

    # 🛡️ Final safety check
    if not should_process_transaction_email(subject, body):
        return {
            "status": "IGNORED",
            "reason": "Email did not pass the transaction security filter"
        }

    message = f"""
Extract the credit card transaction details from this email.

Rules:
- Extract only information explicitly stated.
- Do not guess or invent missing information.
- Amount must be returned as a number without currency symbols.
- If a date is not explicitly present, return None.
- Extract only the last four card digits, never a complete card number.

Subject:
{subject}

Email body:
{body}
"""

    transaction = structured_llm.invoke(message)

    return {
        "status": "PROCESSED",
        "transaction": transaction
    }

In [ ]:
import os

json_files = [f for f in os.listdir() if f.endswith(".json")]

print(json_files)

from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build

SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly"
]

creds = Credentials.from_authorized_user_file(
    "token.json",
    SCOPES
)

gmail_service = build(
    "gmail",
    "v1",
    credentials=creds
)

def build_gmail_query(tracked_cards):
    """
    Build a Gmail search query that looks only for emails
    related to the user's tracked cards.
    """
    return " OR ".join(tracked_cards)


gmail_query = build_gmail_query(TRACKED_CARDS)

print("Searching for:", gmail_query)

results = gmail_service.users().messages().list(
    userId="me",
    q=gmail_query,
    maxResults=20
).execute()

messages = results.get("messages", [])

print(f"Found {len(messages)} matching emails")

def get_email_metadata(message_id):
    message = gmail_service.users().messages().get(
        userId="me",
        id=message_id,
        format="metadata",
        metadataHeaders=["Subject", "From", "Date"]
    ).execute()

    headers = {
        header["name"]: header["value"]
        for header in message.get("payload", {}).get("headers", [])
    }

    return {
        "id": message_id,
        "subject": headers.get("Subject", ""),
        "from": headers.get("From", ""),
        "date": headers.get("Date", "")
    }
for message in messages:
    email = get_email_metadata(message["id"])

    subject = email["subject"]

    # Test transaction classification using subject only for now
    is_transaction = should_process_transaction_email(subject, "")

    if is_transaction:
        status = "✅ PROCESS"
    else:
        status = "⏭️ IGNORE"

    print(f"{status} | {subject}")



['credentials.json', 'token.json']
Searching for: hdfc OR rbl
Found 4 matching emails
⏭️ IGNORE | Fwd: Credit Card xx3353: Higher Loan Amount Update
⏭️ IGNORE | Fwd: your credit card bill payment was successful
✅ PROCESS | Fwd: Alert: Your RBL Bank Credit Card has just been swiped
✅ PROCESS | Fwd: Alert: Your RBL Bank Credit Card has just been swiped
